# Trabajo Práctico: Seguimiento y caracterización de satélites
**Integrantes:** [Franco Diz] y [Lautaro Mesa]  
**Descripción:** Informe de la Etapa 1 enfocado en el análisis estático de la población satelital actual a partir del catálogo CelesTrak.

## 1. Carga de datos y estado inicial

En esta sección se cargan los registros astronómicos desde el archivo JSON, se calcula la fecha media del catálogo (`EPOCH`) para estimar la antigüedad de los datos, y se filtran aquellos objetos cuya altitud es menor a 100 km (considerados decaídos o fuera de órbita operativa).

In [1]:
from datetime import datetime, timezone

from catalogo import leer_satelites
from satelite import crear_satelites
from funciones_base import epoch_medio

# 1. Leer los registros y guardar el momento actual (en UTC)
ruta_json = "datos/satelites.json"
registros = leer_satelites(ruta_json)
ahora = datetime.now(timezone.utc)

# 2. Crear los objetos Satelite descartando los de altitud < 100 km
satelites = crear_satelites(registros, ahora)

# 3. Calcular métricas solicitadas
epoch_dt = epoch_medio(registros)
dias_transcurridos = (ahora - epoch_dt).days
descartados = len(registros) - len(satelites)

# Imprimir resultados
print(f"Registros totales leídos: {len(registros)}")
print(f"Fecha media del EPOCH (datos): {epoch_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Días transcurridos desde el EPOCH medio: {dias_transcurridos} días")
print(f"Satélites descartados (altitud < 100 km): {descartados}")
print(f"Satélites válidos procesados: {len(satelites)}")

Registros totales leídos: 16710
Fecha media del EPOCH (datos): 2026-10-14 06:30:18 UTC
Días transcurridos desde el EPOCH medio: 12 días
Satélites descartados (altitud < 100 km): 16
Satélites válidos procesados: 16694


## 2. Grupos

Se cuenta cuántos satélites pertenece a cada constelación según el prefijo de su nombre, y se calcula qué porcentaje del total es Starlink.


In [2]:
from catalogo import contar_por_grupo

conteo_grupos = contar_por_grupo(registros)
for nombre, cantidad in conteo_grupos.items():
    print(f"{nombre}: {cantidad}")

porcentaje_starlink = conteo_grupos["Starlink"] / len(registros) * 100
print(f"Starlink representa el {porcentaje_starlink:.1f}% del total")

Otros: 4075
GPS: 40
Globalstar: 36
BeiDou: 55
Iridium: 80
OneWeb: 651
Starlink: 11125
Qianfan: 257
Kuiper: 391
Starlink representa el 66.6% del total


## 3. Lanzamientos por año

Cantidad de satélites activos hoy que se lanzaron cada año desde 2015. Es una estimación: no incluye los satélites que ya dejaron de funcionar.

In [3]:
from catalogo import lanzamientos_por_anio

por_anio = lanzamientos_por_anio(registros)
for anio in range(2015, 2027):
    print(f"{anio}: {por_anio.get(anio, 0)}")

anio_max = 2015
for anio in por_anio:
    if anio >= 2015 and por_anio[anio] > por_anio[anio_max]:
        anio_max = anio
print(f"Año con más lanzamientos desde 2015: {anio_max} ({por_anio[anio_max]} satélites)")


2015: 96
2016: 83
2017: 136
2018: 194
2019: 145
2020: 360
2021: 994
2022: 1718
2023: 2266
2024: 2429
2025: 4289
2026: 3285
Año con más lanzamientos desde 2015: 2025 (4289 satélites)


## 4. Tipos de órbita

Clasificación de los satélites según su altitud en el momento del cálculo: LEO (baja), MEO (media), GEO (geoestacionaria) y HEO (alta).

In [4]:
from analisis import contar_por_tipo

conteo_tipos = contar_por_tipo(satelites)
for tipo, cantidad in conteo_tipos.items():
    print(f"{tipo}: {cantidad}")

LEO: 15884
MEO: 202
GEO: 585
HEO: 23


# conclusiòn: 

seccion 1:  el archivo tiene 16.710 satélites y sus órbitas se midieron, en promedio, hace unos 12 días. Esa antigüedad es poca, así que las posiciones calculadas para hoy son confiables (el error crece cuanto más lejos de esa fecha se calcula). Se descartaron unos pocos satélites cuya altitud daba menos de 100 km: ya cayeron a la atmósfera y el cálculo no tiene sentido para ellos. 

seccion 2:  Starlink domina el catálogo: tiene 11.125 satélites, alrededor de dos tercios del total (66,6 %). Después vienen OneWeb (651), Kuiper (391) y Qianfan (257), y el resto de las constelaciones tiene menos de 100 satélites cada una. Hay además 4.075 satélites que no pertenecen a ninguno de estos grupos ("Otros").

seccion 3: la cantidad de satélites lanzados por año creció muchísimo: de 96 en 2015 a 4.289 en 2025, que fue el año con más lanzamientos. El salto fuerte empieza después de 2020, y coincide con el despliegue de las constelaciones grandes. 2026 todavía no terminó, por eso su cantidad (3.285) es parcial. Como el archivo solo tiene los satélites activos hoy, los años más viejos están subestimados.


seccion 4:  la gran mayoría de los satélites está en órbita baja (LEO), alrededor del 95 % del total, lo que es esperable porque Starlink y las otras constelaciones orbitan a pocos cientos de kilómetros. Le siguen la órbita geoestacionaria (GEO), con unos 585 satélites, la media (MEO) con unos 200, y muy pocos en órbita alta (HEO), unos 20.


